# Partial differential equation (PDE) representation (rep)

This notebook demonstrates the application of `PDE` class from `/utils/pde.py`. Conceptually, the class serve as the translator from semi-human readable PDE format to a LISP-like format that is more convenient for the code to base the tensor network construction on.

To initialize the object, we can call the class and provide it with `pde_txt` the semi-human readable PDE and `equal_time_derivative` whether the PDE is equate to zero or first time derivative of the function. For example,

1. $0=5u$: `pde_txt = '5*u'`, `equal_time_derivative = False`,

2. $\partial_tu=5u$: `pde_txt = '5*u'`, `equal_time_derivative = True`.

In [3]:
from utils.pde import PDE

pde_5u = PDE(pde_txt = '5*u', equal_time_derivative = False)

The followings are the grammatical rules required to construct a proper semi-human readble PDE for almost any PDE.

1. As you might already notice, the function that the PDE described is represented with `'u'`. It is a dummy parameter that represent the function regardless of variables and the tensor rank. In other words, $u(x)$, $u(t, x)$, $u(t, x, y, z, ...)$, $\vec{u}(t, x)$, or $u_{ijk}(x)$ are represented with `'u'` in the `pde_txt`. 

2. As shown above, all multiplication must be made explicit with `'*'`.

3. The only allowed binary operations are addition (`'+'`), and multiplication (`'*'`). Subtraction must be made explicit with addition with a term that got multiplied by `-1`. The division by a constant must be represented with the multiplication of the inverse of the constant.

4. The differentiations are assumed to be applyied directly to $u$. Hence, the derivative is represented with simply `'D[  ]'`. The interior of the bracket right after `'D'` depends on the derivative taken. The first argument is the derivative order of $u$ while the optional subsequent arguments are the derivative axes.

5. Any function that is not $u$ must be labeled with `'h_'` followed by a number label indicate the index for the tensor network construction to look for that function. This includes any constant function that is added to other terms, the coordinate functions ($x$, $y$, $x^2y$, ...), and the inverse of any function.

6. The usage of parenthesis (`'('`, `')'`) is allowed as standard mathematical equation.

7. The PDE is assumed to be analytical, i.e., there must be only non-negative power of $u$.

We will go over some examples of these rules, but, first, once the `PDE` object is create we can asked for the parsed version of the PDE as shown below.

In [4]:
pde_5u.pde

['*', 'u', 5.0]

For the one that is set to equal time derivative, the result is as following. Note that the class assume the implicit Euler finite difference method for the time component with inverse time step size `'g'` and the previous time step function is represented with `'p'`. In other words, 

$\partial_tu=5u \rightarrow gu - gp = 5u \rightarrow 0 = 5u - gu + gp$.

In [5]:
pde_5u_eq_t = PDE('5*u', True)
pde_5u_eq_t.pde

['+', ['*', 'u', 5.0], ['*', 'g', 'u', -1.0], ['*', 'g', 'p']]

Since there is nothing special for temporal or spatial derivatives, one can set a coordinate in the derivative to be time as well. We simply provided the option for the fact that the most reliable tensor network optimization algorithm works best with 1D problem. Hence, we need to explicitly remove the time derivative out. However, the platform should work with any number of dimension as long as one has the appropriate amgorithm to solve. For the rest of this demo, we will assume that there is a first order time derivative on the left hand side of the PDE unless explicitly stated otherwise. 

We will go over some more example to showcase the rules established above.

## 1. $\partial_tu = u + 5$.

This seems like the `pde_txt = 'u+5'` situation, but one need to treat $5$ as a constant function and use `pde_txt = 'u+h_0'`. The `PDE` class can parse `'u+5'`, but the subsequent codes will raise an error.

In [6]:
pde_up5_eq_t = PDE('u+h_0', True)
pde_up5_eq_t.pde

['+', 'u', ['h', 0], ['*', 'g', 'u', -1.0], ['*', 'g', 'p']]

## 2. $\partial_tu = 5u + 5$.

As seen at the beginning, eventhough the addition by constant need to be replace with a dummy function of `'h'`, the multiplication by constant is directly allowed.

In [7]:
pde_5up5_eq_t = PDE('5*u+h_0', True)
pde_5up5_eq_t.pde

['+', ['*', 'u', 5.0], ['h', 0], ['*', 'g', 'u', -1.0], ['*', 'g', 'p']]

## 3. $\partial_tu = x^2u$.

In this situation, one can represent $x^2$ as `'h_0*h_1'` where both `'h'`'s represent $x$, but it is better to use as few dummy function as possible, i.e., representing $x^2$ with `'h_0'`.

In [8]:
pde_xsqu_eq_t = PDE('h_0*u', True)
pde_xsqu_eq_t.pde

['+', ['*', ['h', 0], 'u'], ['*', 'g', 'u', -1.0], ['*', 'g', 'p']]

## 4. $\partial_tu = \partial_xu$.

This is the first non-trivial example with some spatial derivative. Since the derivative is first order and the against the first coordinate (assume normal ordering of coordinate $x, y, z, \ldots \rightarrow 0, 1, 2, \ldots$). The derivative term is representing with `'D[1, 0]'` (or `'D[1]'` in the case of 1D problem).

In [9]:
pde_dxu_eq_t = PDE('D[1, 0]', True)
pde_dxu_eq_t.pde

['+', ['D', 1.0, 0.0], ['*', 'g', 'u', -1.0], ['*', 'g', 'p']]

## 5. $0=\nabla u$.

The gradient can be simply represented with the derivative with only the first argument. Since the parser will not check the dimensionality of each term, one need to be careful that the PDE make sense, e.g., the gradient of $u$ is a coveriant vector which cannot be equated with a simple first temporal derivative of scalar function as usual.

In [10]:
pde_du_eq_t = PDE('D[1]', False)
pde_du_eq_t.pde

['+', ['D', 1.0], ['*', 'g', 'u', -1.0], ['*', 'g', 'p']]

## 6. $\partial_tu=\partial_y\partial_x u$.

This is second derivative with the outer one against $y$ while the inner one against $x$. Hence, the `pde_txt = 'D[2, 1, 0]'`.

In [12]:
pde_dydxu_eq_t = PDE('D[2, 1, 0]', True)
pde_dydxu_eq_t.pde

['+', ['D', 2.0, 1.0, 0.0], ['*', 'g', 'u', -1.0], ['*', 'g', 'p']]

## 7. $\partial_tu=u\partial^2_x u$.

This one is the first non-linear example, but the construction process is the same as treating $u$ as any other function.

In [13]:
pde_udxdxu_eq_t = PDE('u*D[2, 0, 0]', True)
pde_udxdxu_eq_t.pde

['+', ['*', 'u', ['D', 2.0, 0.0, 0.0]], ['*', 'g', 'u', -1.0], ['*', 'g', 'p']]

## 8. $\partial_tu=u(\partial_y u)(\partial^2_x u) + (u + x)\partial_zu$.

Even if there are extra terms, the construction are the same in spirit.

In [14]:
pde_udyudxdxup5pxdzu_eq_t = PDE('u*D[1, 1]*D[2, 0, 0] + (u + h_0)*D[1, 2]', True)
pde_udyudxdxup5pxdzu_eq_t.pde

['+',
 ['*', 'u', ['D', 1.0, 1.0], ['D', 2.0, 0.0, 0.0]],
 ['*', ['+', 'u', ['h', 0]], ['D', 1.0, 2.0]],
 ['*', 'g', 'u', -1.0],
 ['*', 'g', 'p']]

## 9. $0=-\partial_tu + u(\partial_y u)(\partial^2_x u) + (u + x)\partial_zu$.

For our last example, we will treat time as additional coordinate. In this case, we will treat time as the first coordinate and all spatial coordinates got bump up by 1.

In [16]:
pde_mdtupudyudxdxup5pxdzu_eq_t = PDE('-1*D[1, 0] + u*D[1, 2]*D[2, 1, 1] + (u + h_0)*D[1, 3]', False)
pde_mdtupudyudxdxup5pxdzu_eq_t.pde

['+',
 ['*', ['D', 1.0, 0.0], -1.0],
 ['*', 'u', ['D', 1.0, 2.0], ['D', 2.0, 1.0, 1.0]],
 ['*', ['+', 'u', ['h', 0]], ['D', 1.0, 3.0]]]

## Final Note

Currently, there is no proper way to handle some important tersorial object like Kronecker delta, and Levi-Civita symbol which implies that the object cannot handle divergent, and curl since there will not appear in the 1D problem we will show in these demos. This might be some nice extension to the project.